# Transform Orders
Transforms the orders data and saves the cleaned result al a Silver table.

In [0]:
from pyspark.sql import functions as F


In [0]:
df = (
    spark.read
    .table("ecommerce_catalog.bronze.orders_raw")
    .withColumn(
                "is_imputed", 
                F.lit(None).cast("string")
                )
    .withColumn( 
                "order_date",
                F.coalesce(
                    F.try_to_date("order_date", "yyyy-MM-dd"),
                    F.try_to_date("order_date", "MM/dd/yyyy"),
                    F.try_to_date("order_date", "dd.MM.yyyy")
                        )
                    )
)



display(df)


In [0]:
id_columns = ["order_id", "customer_id", "product_id"]
for column_name in id_columns:
    df = df.withColumn(
        column_name,
        F.upper(F.trim(F.col(column_name)))
    )

In [0]:


display(df.filter(F.col("product_id").isNull()).dropDuplicates(["unit_price"]))



In [0]:
price_product_mapping = (
    df.filter(
        F.col("product_id").isNotNull()
        & F.col("unit_price").isNotNull()
    )
    .groupBy("unit_price")
    .agg(
        F.countDistinct("product_id").alias("product_count"),
        F.first("product_id").alias("mapped_product_id")
    )
    .filter(F.col("product_count") == 1)
)
display(price_product_mapping)



In [0]:
df = (
    df.alias("orders")
    .join(
        price_product_mapping.alias("mapping"),
        on = F.col("orders.unit_price"),
        how = "left"
    )
    .withColumn(
        "is_imputed",
        F.when(
            F.col("orders.product_id").isNull()
            & F.col("mapping.mapped_product_id").isNotNull(),
            F.lit("product_id")
        ).otherwise(F.col("is_imputed"))
    )
    .withColumn(
        "product_id",
        F.coalesce(
            F.col("orders.product_id"),
            F.col("mapping.mapped_product_id")
        )
    )
    .drop("product_count", "mapped_product_id")
)
display(df)


In [0]:
display(df.select(F.col("product_id")))

In [0]:
display(df.filter(F.col("unit_price").isNull()))

In [0]:
display(df.filter(F.col("unit_price")<0))
display(df.filter(F.col("unit_price")==0))
display(df.orderBy(F.desc("unit_price")).limit(50))

In [0]:
high_price=["P0111",
"P0103",
"P0001",
"P0106"]
display(df.filter(F.col("product_id").isin(high_price)).orderBy("product_id"))

In [0]:



invalid_condition = (
    F.col("unit_price").isNull()
    | F.col("unit_price").isin([0.0, 99999.99])
    | (F.col("unit_price") < 0)
)


invalid_product_ids = [
    row.product_id
    for row in df.filter(invalid_condition)
    .select("product_id")
    .distinct()
    .collect()
]


result_df = (
    df.filter(F.col("product_id").isin(invalid_product_ids))
    .groupBy("product_id")
    .agg(F.mode("unit_price").alias("unit_price"))
)

display(result_df)

### **Observation:**

For products with invalid price values (`NULL`, 0, negative values, or obvious outliers), the price distribution was analyzed. For each affected `product_id`, the most frequently occurring valid price was determined.

### **Action:**

The most frequent valid price serves as a reference value for the imputation or correction of invalid prices, as it represents the most frequently observed value for this product in the dataset.

In [0]:
from pyspark.sql import functions as F


invalid_condition = (
    F.col("unit_price").isNull()
    |F.col("unit_price").isin([0.0, 99999.99])
    |(F.col("unit_price") < 0)
)


replacement_df = result_df.withColumnRenamed("unit_price", "imputed_unit_price")


df = (
    df.join(replacement_df, on="product_id", how="left")
    .withColumn(
        "is_imputed",
        F.when(
            invalid_condition,
            F.lit("unit_price")
        ).otherwise(F.col("is_imputed") if "is_imputed" in df.columns else F.lit(None))
    )
    .withColumn(
        "unit_price",
        F.when(
            invalid_condition,
            F.col("imputed_unit_price")
        ).otherwise(F.col("unit_price"))
    )
    .drop("imputed_unit_price")
)

display(df)

In [0]:
display(df.filter(F.col("unit_price")<0))
display(df.filter(F.col("unit_price")==0))
display(df.filter(F.col("unit_price").isNull()))
display(df.orderBy(F.desc("unit_price")).limit(5))

In [0]:
duplicate_values = (df.groupBy("order_id") 
                        .count() 
                        .filter(F.col("count") > 1)
)

display(duplicate_values)

In [0]:

duplicate_list = [
    row.order_id
    for row in df.groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .select("order_id")
    .collect()
]

print(duplicate_list)

In [0]:
count = 0
for i in duplicate_list:
    if count == 5:
        break
    display(df.filter(F.col("order_id") == i))
    count += 1


In [0]:

df = df.dropDuplicates()

duplicate_values = (df.groupBy("order_id") 
                        .count() 
                        .filter(F.col("count") > 1)
)

display(duplicate_values)

In [0]:
display(df.filter(F.col("customer_id").isNull()))

In [0]:
null_customer_dates = [
    row.order_date
    for row in df.filter(F.col("customer_id").isNull())
                 .select("order_date")
                 .collect()
]

filtered_df = (df.filter(F.col("order_date")
                         .isin(null_customer_dates))
                .orderBy("order_date"))    


display(filtered_df)

In [0]:
duplicate_keys = (df.groupBy("customer_id") 
                   .count() 
                   .filter(F.col("count") > 1) 
                   .select("customer_id"))


duplicates_df = (df.join(duplicate_keys, 
                         on="customer_id", 
                         how="inner") 
                  .orderBy("customer_id"))


display(duplicates_df)

### **Observation:**

Records with missing `customer_id` values were analyzed against available attributes (e.g., `order_date`, `product_id`, `quantity`, and other fields). However, an unambiguous mapping to an existing `customer_id` could not be established.

### **Action:**

Since no reliable reconstruction is possible, the missing `customer_id` values are intentionally not imputed. The remaining `NULL` values are retained and documented.

In [0]:

df = df.select(target_column_order)

display(df)

In [0]:
display(df.filter(F.col("is_imputed").isNotNull()))

In [0]:
from delta.tables import DeltaTable

if sparks.catalog.tableExists("ecommerce_catalog.silver.silver_orders"):
   dlt_obj = DeltaTable.forName(spark, "ecommerce_catalog.silver.silver_orders")

   (
       dlt_obj.alias("trg").merge(df.alias("src"),"trg.order_id == src.order_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
        )

else:
    (
        df.write.format("delta")
        .mode("append")\
        .saveAsTable("ecommerce_catalog.silver.silver_orders")
        )



### **Summary**
In the Silver layer, raw data was standardized and cleaned. Date formats were unified, complete duplicates were removed, missing or invalid values were imputed where unambiguously possible, and text fields were standardized.

Values that could not be unambiguously reconstructed—specifically missing `customer_id` values—were intentionally retained unchanged to avoid introducing potentially inaccurate information into the dataset.

The resulting table serves as the foundation for business transformations in the Gold layer.
